# WTI vs DXY — Cointegration and Standard-Deviation Analysis

This notebook examines **two different meanings of “use standard deviation instead of raw price”**:

**A. Standardised price levels:** transform WTI and DXY to whole-sample Z-scores (units of standard deviations). This removes unit differences, **but does not change whether the levels are cointegrated**, because a fixed affine rescaling leaves cointegration intact.

**B. Rolling volatility:** compute the rolling standard deviation of daily WTI price changes and daily DXY percentage returns. This asks whether their *volatility series* share a relationship, which is **not** the same as cointegration of prices. We test the integration order before deciding whether cointegration is appropriate.

**Data warning:** Yahoo Finance `CL=F` is a front-month futures history, not a true constant-maturity or spot WTI series. Roll gaps and the April 2020 negative settlement can distort daily changes and volatility. Interpret results as exploratory, not publication-ready.

## 1 · Setup and configuration

In [ ]:
# Uncomment if needed:
# %pip install -q yfinance pandas numpy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

from statsmodels.tsa.stattools import adfuller, coint
from IPython.display import display

START = "2010-01-01"
END = None                  # None -> latest data Yahoo provides
WINDOW = 30                 # rolling volatility window (trading days)
MIN_OBS = 100               # minimum observations for a series test
ALPHA = 0.05

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

## 2 · Download and align WTI and DXY

In [ ]:
tickers = {"WTI": "CL=F", "DXY": "DX-Y.NYB"}
series = {}

for label, ticker in tickers.items():
    history = yf.download(ticker, start=START, end=END,
                          auto_adjust=True, progress=False)
    if history.empty:
        raise RuntimeError(f"No observations returned for {ticker}. Check the symbol or connection.")
    close = history["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    series[label] = pd.to_numeric(close, errors="coerce").rename(label)

prices = pd.concat(series.values(), axis=1).replace([np.inf, -np.inf], np.nan).dropna()
prices = prices.loc[~prices.index.duplicated(keep="first")].sort_index()
assert len(prices) >= MIN_OBS, "Too few aligned daily observations."

print(f"Aligned observations: {len(prices):,} | {prices.index.min().date()} to {prices.index.max().date()}")
display(prices.head())

ax = prices.plot(subplots=True, figsize=(12, 6), title=["WTI front-month futures ($/bbl)", "DXY (index points)"], legend=False)
plt.tight_layout()
plt.show()

## 3 · Statistical helper functions

The ADF null hypothesis is a unit root. Failure to reject **does not prove** a unit root. The Engle–Granger null hypothesis is **no cointegration** and its p-value already accounts for fitting the regression; do not substitute an ordinary ADF p-value on fitted residuals.

In [ ]:
def adf_summary(s, label):
    s = pd.Series(s).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < MIN_OBS:
        return {"series": label, "n": len(s), "ADF statistic": np.nan, "ADF p": np.nan}
    stat, p, *_ = adfuller(s, autolag="AIC", regression="c")
    return {"series": label, "n": len(s), "ADF statistic": stat, "ADF p": p}


def integration_screen(frame):
    rows = []
    for label in frame.columns:
        rows.append(adf_summary(frame[label], label + " levels"))
        rows.append(adf_summary(frame[label].diff(), label + " first difference"))
    return pd.DataFrame(rows).set_index("series")


def eg_test(frame, y, x, label):
    pair = frame[[y, x]].dropna()
    if len(pair) < MIN_OBS:
        print(f"{label}: insufficient data for cointegration test")
        return None
    fit = sm.OLS(pair[y], sm.add_constant(pair[x])).fit()
    eg_stat, eg_p, critical = coint(pair[y], pair[x], trend="c", autolag="aic")
    residual = pair[y] - fit.predict(sm.add_constant(pair[x]))
    print(f"\n{label}")
    print(f"y = {fit.params['const']:.4f} + ({fit.params[x]:.4f}) * x")
    print(f"R²: {fit.rsquared:.4f} | Engle–Granger stat: {eg_stat:.4f} | p: {eg_p:.6g}")
    print("Critical values (1%, 5%, 10%):", np.round(critical, 4))
    print("Decision:", "Reject no cointegration" if eg_p < ALPHA else "Fail to reject no cointegration")
    return {"model": fit, "residual": residual, "p": eg_p,
            "test_stat": eg_stat, "critical_values": critical}


def zscore(s):
    # A FIXED whole-sample mean and standard deviation (descriptive, not OOS trading-safe)
    sd = s.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError(f"Series {s.name} has zero/invalid standard deviation")
    return (s - s.mean()) / sd

## 4 · A: Cointegration of prices expressed in standard deviations

Transform each *level* as \(Z^X_t=(X_t-\bar X)/s_X\). Both series now measure deviations from their sample means in standard-deviation units. **This rescaling does not create cointegration:** the Engle–Granger test statistic should be (up to numerical precision) the same as for raw prices.

In [ ]:
price_z = prices.apply(zscore)

display(integration_screen(price_z))
raw_result = eg_test(prices, "WTI", "DXY", "Raw price levels (reference)")
z_result = eg_test(price_z, "WTI", "DXY", "Standardised price levels (Z-scores)")

if raw_result is not None and z_result is not None:
    print("Absolute difference between EG p-values:", abs(raw_result["p"] - z_result["p"]))

price_z.plot(title="WTI and DXY levels in standard-deviation units", ylabel="Whole-sample Z-score")
plt.axhline(0, color="black", ls="--", lw=1)
plt.tight_layout()
plt.show()

if z_result is not None:
    z_result["residual"].plot(title="Standardised-price OLS equilibrium residual", ylabel="Residual (standardised units)")
    plt.axhline(0, color="black", ls="--", lw=1)
    plt.tight_layout()
    plt.show()

## 5 · B: Rolling standard deviation of daily changes (volatility)

For WTI, use \(\Delta WTI_t=WTI_t-WTI_{t-1}\), in **USD/barrel per day**, avoiding percentage-return explosions near the historically negative futures price. For DXY use \(r_t^{DXY}=100(DXY_t/DXY_{t-1}-1)\), in **percent per day**.

Then compute a 30-day rolling **sample standard deviation** of each change series. Finally, standardise both volatility series into Z-score units to make the plot comparable. This is **volatility**, not standardised price, and has a different interpretation.

In [ ]:
changes = pd.DataFrame(index=prices.index)
changes["WTI_change_USD"] = prices["WTI"].diff()
changes["DXY_return_pct"] = prices["DXY"].pct_change(fill_method=None) * 100

vol = pd.DataFrame(index=prices.index)
vol["WTI_vol"] = changes["WTI_change_USD"].rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
vol["DXY_vol"] = changes["DXY_return_pct"].rolling(WINDOW, min_periods=WINDOW).std(ddof=1)
vol = vol.dropna()

# Their magnitudes are not comparable in native units; convert to Z-scores.
vol_z = vol.apply(zscore)
print(f"Volatility window: {WINDOW} trading days | aligned volatility observations: {len(vol):,}")
display(vol.head())

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 7))
vol.plot(ax=axes[0], subplots=False, secondary_y="DXY_vol", title="30-day rolling standard deviations (native units)")
vol_z.plot(ax=axes[1], title="Volatility series expressed in standard-deviation units")
axes[1].axhline(0, color="black", ls="--", lw=1)
plt.tight_layout()
plt.show()

## 6 · Determine whether a cointegration test makes sense for volatility

Cointegration is normally a relationship between \(I(1)\) variables. Volatility series are often persistent but stationary, so **a small cointegration p-value on stationary volatility is not evidence of the usual long-run price-level equilibrium**. We report the tests, but only interpret Engle–Granger as cointegration when both series look \(I(1)\) in preliminary screening.

In [ ]:
print("ADF tests on rolling volatility (native units)")
vol_adf = integration_screen(vol)
display(vol_adf)

vol_level_p = vol_adf.loc[["WTI_vol levels", "DXY_vol levels"], "ADF p"]
vol_diff_p = vol_adf.loc[["WTI_vol first difference", "DXY_vol first difference"], "ADF p"]

both_candidate_I1 = bool((vol_level_p > ALPHA).all() and (vol_diff_p < ALPHA).all())
both_stationary = bool((vol_level_p < ALPHA).all())

if both_candidate_I1:
    print("Both rolling volatility series look like I(1) candidates; running Engle–Granger.")
    vol_eg = eg_test(vol_z, "WTI_vol", "DXY_vol", "Rolling-volatility Z-score cointegration")
elif both_stationary:
    print("Both rolling volatility series reject a unit root at 5%.")
    print("Cointegration of I(1) series is not the appropriate framework here.")
    vol_eg = None
else:
    print("ADF results are mixed or inconclusive; no reliable I(1) case established.")
    print("Do not infer cointegration from these volatility series without further unit-root checks.")
    vol_eg = None

## 7 · Volatility co-movement: correlation and HAC-robust regression

Whether or not volatility is \(I(1)\), it is informative to inspect contemporaneous associations between the **standardised volatility series**. The regression below uses Newey–West/HAC standard errors to reduce sensitivity to serial correlation. It is **descriptive, not causal**, and rolling windows overlap heavily.

In [ ]:
corr = vol_z["WTI_vol"].corr(vol_z["DXY_vol"])
fit_hac = sm.OLS(
    vol_z["WTI_vol"], sm.add_constant(vol_z["DXY_vol"])
).fit(cov_type="HAC", cov_kwds={"maxlags": WINDOW})

print(f"Contemporaneous Pearson correlation of rolling volatilities: {corr:.4f}")
print(fit_hac.summary())

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(vol_z["DXY_vol"], vol_z["WTI_vol"], s=7, alpha=0.25)
xx = np.linspace(vol_z["DXY_vol"].min(), vol_z["DXY_vol"].max(), 100)
ax.plot(xx, fit_hac.params["const"] + fit_hac.params["DXY_vol"]*xx, color="red", lw=2)
ax.set(xlabel="DXY rolling-volatility Z-score", ylabel="WTI rolling-volatility Z-score",
       title="Rolling volatility association")
plt.tight_layout()
plt.show()

## 8 · Optional rolling correlation and exports

A single full-sample association may hide changes in behaviour. The rolling correlation below is *descriptive*; it does not establish structural breaks or causality.

In [ ]:
ROLLING_CORR = 252
rolling_corr = vol_z["WTI_vol"].rolling(ROLLING_CORR).corr(vol_z["DXY_vol"])
rolling_corr.plot(title=f"{ROLLING_CORR}-day rolling correlation: WTI vs DXY volatility",
                  ylabel="Correlation")
plt.axhline(0, color="black", ls="--", lw=1)
plt.tight_layout()
plt.show()

output = prices.join(price_z.add_suffix("_price_z"))
output = output.join(changes).join(vol).join(vol_z.add_suffix("_vol_z"))
output["rolling_vol_corr"] = rolling_corr
output.to_csv("wti_dxy_std_analysis.csv", index_label="Date")

print("Saved wti_dxy_std_analysis.csv")
print("\nREADOUT")
print("1) Z-scoring raw prices cannot change their cointegration status.")
print("2) ADF on rolling volatility controls whether cointegration is an appropriate question.")
print("3) Correlation/HAC on volatility measures co-movement, not a stable price equilibrium.")
print("4) For research, repeat with spot/constant-maturity oil data and sensitivity to window size/rolls.")